# Day 3 Session 1: Likelihood and Random Walk Intuition

By the end of this session, participants should be able to:

1. Explain likelihood in plain language.
2. Calculate a likelihood score for a simple normal model.
3. Visualize a likelihood curve over possible parameter values.
4. Describe a random walk as a sequence of dependent random moves.
5. Connect grid search, gradient search, and MCMC conceptually.


> **Participant notebook:** Work through the examples in order. For exercises that ask you to modify an existing example, a copied workspace or starter cell is included so you can change the named values without editing the original demonstration cell.


## Big picture

On Day 2, we used **grid search** and **gradient search** to find parameter values that make the model fit the data well.

Today we introduce another idea:

> Instead of only finding one best parameter value, we can explore many reasonable parameter values.

Before we do MCMC, we need a way to score parameter values. That score will come from the **likelihood function**.

In this workshop, we will keep the statistics simple:

- We focus on the **normal distribution**.
- We estimate only one parameter, the mean $\mu$.
- We fix the standard deviation $\sigma$.
- We avoid detailed Bayesian terminology.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Use a clean random number generator for reproducibility.
rng = np.random.default_rng(42)

## 1. A normal curve as a model for data

Suppose we observe one value:

$$
y = 5.1.
$$

We want to compare two possible explanations:

- Model A: the data came from a normal distribution centered at $\mu=2$.
- Model B: the data came from a normal distribution centered at $\mu=5$.

Intuitively, $y=5.1$ is more consistent with $\mu=5$ than with $\mu=2$.

For a normal model,

$$
Y \sim N(\mu, \sigma^2),
$$

the probability density function is

$$
f(y\mid \mu, \sigma)
=
\frac{1}{\sigma\sqrt{2\pi}}
\exp\left(-\frac{(y-\mu)^2}{2\sigma^2}\right).
$$

For today, we treat this density value as a **score** for how well the parameter value $\mu$ explains the observation.


In [ ]:
def normal_pdf(y, mu, sigma):
    """Normal probability density function."""
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((y - mu) / sigma) ** 2)

# One observed data point
observed_y = 5.1
sigma = 0.5

mu_a = 2.0
mu_b = 5.0

score_a = normal_pdf(observed_y, mu_a, sigma)
score_b = normal_pdf(observed_y, mu_b, sigma)

print(f"Observed value: y = {observed_y}")
print(f"Density score if mu = {mu_a}: {score_a:.6f}")
print(f"Density score if mu = {mu_b}: {score_b:.6f}")
print(f"The score for mu = {mu_b} is much larger, so it explains the observation better.")

x = np.linspace(0, 7, 400)
plt.figure(figsize=(8, 4))
plt.plot(x, normal_pdf(x, mu_a, sigma), label=r"Normal curve with $\mu=2$")
plt.plot(x, normal_pdf(x, mu_b, sigma), label=r"Normal curve with $\mu=5$")
plt.axvline(observed_y, linestyle="--", label="Observed value")
plt.scatter([observed_y], [normal_pdf(observed_y, mu_a, sigma)])
plt.scatter([observed_y], [normal_pdf(observed_y, mu_b, sigma)])
plt.xlabel("Possible observation value")
plt.ylabel("Density")
plt.title("Which normal curve better explains the observation?")
plt.legend()
plt.show()

### Exercise 1: Normal curve interpretation

Change the value of `observed_y` in the code above.

Try:

- `observed_y = 2.2`
- `observed_y = 3.5`
- `observed_y = 5.1`

Discuss:

1. Which value of $\mu$ gives the larger density score?
2. Does the graph agree with your intuition?
3. What happens if you make `sigma` larger, such as `sigma = 1.5`?


In [ ]:
# Copied workspace for Exercise 1.
# Change observed_y or sigma, then rerun.
observed_y_exercise = 5.1
sigma_exercise = 0.5

score_a = normal_pdf(observed_y_exercise, mu_a, sigma_exercise)
score_b = normal_pdf(observed_y_exercise, mu_b, sigma_exercise)
print(f"Density score if mu = {mu_a}: {score_a:.6f}")
print(f"Density score if mu = {mu_b}: {score_b:.6f}")


## 2. From one observation to several observations

Now suppose we observe several values:

$$
y_1, y_2, \ldots, y_n.
$$

If we assume the observations are independent and normally distributed, then the likelihood is

$$
L(\mu)
=
f(y_1\mid\mu,\sigma)
f(y_2\mid\mu,\sigma)
\cdots
f(y_n\mid\mu,\sigma).
$$

In words:

> The likelihood multiplies the model's score for each observed data point.

Because multiplying many small numbers can cause numerical problems, we usually use the **log-likelihood**:

$$
\ell(\mu)
=
\log L(\mu).
$$

Since log changes multiplication into addition,

$$
\ell(\mu)
=
\sum_{i=1}^n \log f(y_i\mid \mu,\sigma).
$$


In [ ]:
# A small set of observations.
# You can think of these as repeated measurements from a classroom activity or science experiment.
y_data = np.array([4.8, 5.1, 5.0, 5.3, 4.9, 5.2, 4.7, 5.0])
sigma = 0.3

print("Observed data:", y_data)
print(f"Sample mean: {np.mean(y_data):.3f}")

plt.figure(figsize=(8, 3))
plt.hist(y_data, bins=5, edgecolor='black', alpha=0.7)
plt.yticks([])
plt.xlabel("Observed value")
plt.title("Observed data points")
plt.show()

In [ ]:
def log_likelihood_mu(mu, y, sigma):
    """Log-likelihood for a normal model with unknown mean mu and fixed sigma."""
    log_density_values = -np.log(sigma * np.sqrt(2 * np.pi)) - 0.5 * ((y - mu) / sigma) ** 2
    return np.sum(log_density_values)

candidate_mus = [4.0, 5.0, 6.0]

for mu in candidate_mus:
    ll = log_likelihood_mu(mu, y_data, sigma)
    print(f"mu = {mu:.1f}, log-likelihood = {ll:.3f}")

## 3. Visualizing the likelihood curve

Now we evaluate many possible values of $\mu$. This is very close to a one-dimensional grid search.

The largest point on the curve is the parameter value that best explains the observed data according to this model.


In [ ]:
mu_grid = np.linspace(3.5, 6.5, 300)
ll_values = np.array([log_likelihood_mu(mu, y_data, sigma) for mu in mu_grid])

best_index = np.argmax(ll_values)
best_mu = mu_grid[best_index]

plt.figure(figsize=(8, 4))
plt.plot(mu_grid, ll_values)
plt.axvline(best_mu, linestyle="--", label=f"Best grid value: {best_mu:.3f}")
plt.axvline(np.mean(y_data), linestyle=":", label=f"Sample mean: {np.mean(y_data):.3f}")
plt.xlabel(r"Candidate mean $\mu$")
plt.ylabel("Log-likelihood")
plt.title("Likelihood as an objective function")
plt.legend()
plt.show()

print(f"Best mu from grid search: {best_mu:.3f}")
print(f"Sample mean: {np.mean(y_data):.3f}")

### Discussion pause

This graph helps connect Day 2 and Day 3:

- **Grid search** checks many points and chooses the largest likelihood.
- **Gradient search** tries to climb the likelihood curve more efficiently.
- **MCMC** will take a random walk that spends more time near high-likelihood values.

A useful classroom phrase:

> Likelihood gives us a way to score parameter values based on the data we observed.


### Exercise 2: Likelihood curve

Change the dataset or the value of `sigma`.

Try:

```python
y_data = np.array([4.8, 5.1, 5.0, 5.3, 4.9, 5.2, 4.7, 5.0])
sigma = 0.3
```

then try:

```python
y_data = np.array([4.8, 5.1, 5.0, 5.3, 4.9, 7.0])
sigma = 0.3
```

Discuss:

1. How does one unusual observation change the likelihood curve?
2. What happens when `sigma` is small?
3. What happens when `sigma` is large?


In [ ]:
# Copied workspace for Exercise 2.
# Change y_data_exercise or sigma_exercise, then rerun.
y_data_exercise = np.array([4.8, 5.1, 5.0, 5.3, 4.9, 5.2, 4.7, 5.0])
sigma_exercise = 0.3

mu_grid_exercise = np.linspace(3.5, 6.5, 300)
ll_values_exercise = np.array([
    log_likelihood_mu(mu, y_data_exercise, sigma_exercise)
    for mu in mu_grid_exercise
])
best_mu_exercise = mu_grid_exercise[np.argmax(ll_values_exercise)]
print(f"Best exercise mu: {best_mu_exercise:.3f}")


## 4. Random walk intuition

Before introducing MCMC, we first look at a simple random walk.

Start at a current position. At each step, move left or right by a random amount.

A simple version is

$$
x_{t+1} = x_t + \epsilon_t,
$$

where

$$
\epsilon_t \sim N(0, s^2).
$$

Here $s$ controls the step size.


In [ ]:
def simulate_random_walk(start=0.0, n_steps=100, step_size=1.0, seed=123):
    """Simulate a one-dimensional random walk."""
    local_rng = np.random.default_rng(seed)
    positions = np.zeros(n_steps + 1)
    positions[0] = start
    for t in range(n_steps):
        step = local_rng.normal(0, step_size)
        positions[t + 1] = positions[t] + step
    return positions

positions = simulate_random_walk(start=0.0, n_steps=100, step_size=0.5)

plt.figure(figsize=(8, 4))
plt.plot(positions, marker="o", markersize=3)
plt.axhline(0, linestyle="--")
plt.xlabel("Step number")
plt.ylabel("Position")
plt.title("A simple random walk")
plt.show()

### Exercise 3: Random walk step size

Change `step_size` in the code above.

Try:

- `step_size = 0.1`
- `step_size = 0.5`
- `step_size = 2.0`

Discuss:

1. When the step size is small, does the walk explore quickly or slowly?
2. When the step size is large, does the walk jump around more?
3. Why might step size matter for MCMC?


In [ ]:
# Copied workspace for Exercise 3.
# Change step_size_exercise, then rerun.
step_size_exercise = 0.5
positions_exercise = simulate_random_walk(start=0.0, n_steps=100, step_size=step_size_exercise)

plt.figure(figsize=(8, 4))
plt.plot(positions_exercise, marker="o", markersize=3)
plt.axhline(0, linestyle="--")
plt.xlabel("Step number")
plt.ylabel("Position")
plt.title(f"Random walk with step size {step_size_exercise}")
plt.show()


## 5. The Markov chain idea

A Markov chain is a random process where the next state depends on the current state.

The key idea is:

> The next move depends on where we are now, not on the entire history.

The random walk is a Markov chain because the next position is generated from the current position.

Here is a tiny two-state Markov chain example reused from the earlier Markov chain material.


In [ ]:
# Two-state Markov chain example
# State 0 and State 1 could represent any two categories.

PX = np.array([0.0, 1.0])  # Starting distribution: definitely in State 1
T = np.array([[0.7, 0.3],
              [0.3, 0.7]])  # Transition matrix

history = [PX.copy()]
for t in range(20):
    PX = PX @ T
    history.append(PX.copy())

history = np.array(history)

plt.figure(figsize=(8, 4))
plt.plot(history[:, 0], marker="o", label="Probability of State 0")
plt.plot(history[:, 1], marker="o", label="Probability of State 1")
plt.xlabel("Step")
plt.ylabel("Probability")
plt.title("A simple Markov chain approaches a stable pattern")
plt.legend()
plt.show()

print("Final distribution after 20 steps:", history[-1])

## Session 1 wrap-up

We now have the ingredients for MCMC:

1. A **likelihood function** that scores parameter values.
2. A **random walk proposal** that suggests a new parameter value.
3. A **Markov chain** that moves from the current value to the next value.

In Session 2, we combine these ideas:

> MCMC = random walk proposals + likelihood-based accept/reject decisions.
